[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# An Event Store


## What you will be able to do

Build one small program that uses both drivers on purpose and can say why each part got the one it
got: psycopg for the `COPY` load and the pipelined summary writes, asyncpg for the concurrent
readers and the listener. Make it survive the failure that only appears once those pieces are put
together, which is a thousand rows arriving as a single notification. And reconcile with a watermark
rather than a counter, so that nothing is lost when the listener was disconnected, woken once, or
woken twice.


## The idea

### The problem

Every piece of this guide works. Put four of them in one program and a new failure appears that none
of the four has on its own: the loader writes a thousand rows in one transaction, the trigger raises
a thousand notifications, and the listener is woken exactly once, because PostgreSQL collapses
identical notifications inside a transaction into one delivery.

A worker that treats a notification as "one row arrived" loses nine hundred and ninety-nine of them
and reports no error at all.

### What an event store is

An append-only table of things that happened, a summary built from it, and readers that ask
questions of both. Nothing is updated in place, so the only question a reader or a follower ever has
is "what is new since the last one I saw", which is the watermark.

### Why two drivers

Because this guide measured them rather than guessing. `COPY` and pipeline mode are psycopg
features with no asyncpg equivalent, and the load and the summary writes are exactly those two
shapes. Concurrent readers and a listener are where asynchronous code pays, and asyncpg has no
autocommit to forget on a listener. **Which Driver** is why those sentences are claims rather than
preferences.

### Where this shows up

Any system with ingestion on one side and queries on the other, which is most of them. The
watermark in particular is the pattern that makes a message you might miss safe to build on.

### What this notebook covers

The four pieces, each with a line saying which driver it uses and why. Then the whole program. Then
the failures that only exist once they are assembled: the collapsed notification, the pool that is
smaller than the number of tasks, two summary writers colliding, and the messages sent while nobody
was listening.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import threading

import psycopg


def load_a_thousand():
    with psycopg.connect("dbname=guide") as writer:
        with writer.cursor().copy("COPY store (kind, payload) FROM STDIN") as copy:
            for number in range(1000):
                copy.write_row(("click", '{"n": %d}' % number))
        writer.commit()                              # one transaction, a thousand rows


listener = psycopg.connect("dbname=guide", autocommit=True)
listener.execute("TRUNCATE store")
listener.execute("LISTEN store")

threading.Timer(0.2, load_a_thousand).start()
woken = list(listener.notifies(timeout=5, stop_after=5))

print("rows loaded:   ", listener.execute("SELECT count(*) FROM store").fetchone()[0])
print("times woken up:", len(woken))
print("a worker that counted notifications would have lost 999 rows")
listener.close()
```

```
rows loaded:    1000
times woken up: 1
a worker that counted notifications would have lost 999 rows
```

A row trigger that fired a thousand times, and one delivery. PostgreSQL collapses duplicate
notifications within a transaction, and a thousand identical ones are duplicates. Nothing here is
broken and nothing raised: the program is simply wrong if it believed the count.


## Setup

Thirteen imports, both drivers, psycopg's pool, the server, and the store.

- `psycopg` and `psycopg_pool` for the load, the summary and the pool, `asyncpg` for the rest
- `asyncio` runs the asynchronous half, `threading` loads beside a listener, `time` measures
- `json` builds the payloads and `logging` quiets the pool's retries
- `subprocess`, `sys`, `os`, `getpass` stand the server up with `version` and `PackageNotFoundError`

`build_store` makes the `store` and `summary` tables and the trigger, and empties them, so the
notebook can be run from the top as many times as you like. `events` makes rows to load, and
`counted` asks a one-line question on a connection of its own.


In [1]:
import asyncio
import getpass
import json
import logging
import os
import subprocess
import sys
import threading
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
import psycopg_pool

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

logging.getLogger("psycopg.pool").setLevel(logging.CRITICAL)        # its retries are not the lesson


def build_store():
    """The two tables and the trigger this notebook is about. Idempotent, and it empties them."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS store (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS summary (
                            kind text PRIMARY KEY,
                            n bigint NOT NULL,
                            through bigint NOT NULL)""")            # the watermark, stored
        conn.execute("""CREATE OR REPLACE FUNCTION store_landed() RETURNS trigger AS $$
                        BEGIN
                            PERFORM pg_notify('store', NEW.kind);
                            RETURN NEW;
                        END;
                        $$ LANGUAGE plpgsql""")
        conn.execute("""CREATE OR REPLACE TRIGGER store_announced AFTER INSERT ON store
                        FOR EACH ROW EXECUTE FUNCTION store_landed()""")
        conn.execute("TRUNCATE store, summary")
        return "store, summary and the trigger are ready"


def events(count, kind="click", start=0):
    """Rows to load, as tuples, which is what COPY wants."""
    return [(kind, json.dumps({"n": number, "size": 1 + number % 7}))
            for number in range(start, start + count)]


def counted(sql="SELECT count(*) FROM store", parameters=None):
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        return conn.execute(sql, parameters).fetchone()[0]


print("server:", start_server())
print("events table:", build(), "rows")
print(build_store())


server: already running
events table: 5000 rows
store, summary and the trigger are ready


## Worked examples

### The load: psycopg, because `COPY` is psycopg's

Ten thousand rows, in one statement, from Python objects:


In [2]:
def load(rows):
    """psycopg, because COPY has no asyncpg equivalent and this is the fastest way in."""
    with psycopg.connect("dbname=guide") as conn:
        with conn.cursor().copy("COPY store (kind, payload) FROM STDIN") as copy:
            for row in rows:
                copy.write_row(row)
        conn.commit()


load(events(10_000))
print("in the table:", counted())
print("one statement, one transaction, one commit")


in the table: 10000
one statement, one transaction, one commit


`COPY` is one statement carrying every row, which is why **COPY** measured it as an order of
magnitude faster than a loop of inserts. asyncpg has `copy_records_to_table`, which is the same idea,
and psycopg's streaming interface is the one this guide taught, so this half stays where it started.

### The summary: psycopg, because pipeline mode is psycopg's

Many small writes, with the round trips taken out:


In [3]:
def summarize():
    """psycopg, because a pipeline removes a round trip per statement and asyncpg has none."""
    with psycopg.connect("dbname=guide") as conn:
        high = conn.execute("SELECT coalesce(max(id), 0) FROM store").fetchone()[0]
        totals = conn.execute("SELECT kind, count(*) FROM store WHERE id <= %s GROUP BY kind",
                              (high,)).fetchall()

        with conn.pipeline():                                       # queued, not one at a time
            for kind, number in totals:
                conn.execute("""INSERT INTO summary (kind, n, through) VALUES (%s, %s, %s)
                                ON CONFLICT (kind) DO UPDATE
                                SET n = EXCLUDED.n, through = EXCLUDED.through""",
                             (kind, number, high))
        conn.commit()
    return high, totals


through, totals = summarize()
print("summarized through id", through, ":", totals)


summarized through id 10000 : [('click', 10000)]


Two things are worth pointing at. The `max(id)` is read first and every later statement is bounded
by it, so the summary describes a definite prefix of the table rather than whatever happened to be
there when each statement ran. And `through` is written down, which is what makes the next run able
to say what it has not seen.

### The readers: asyncpg, because they run at the same time

Several independent questions, each on its own connection:


In [4]:
async def questions(pool):
    """asyncpg, because these overlap and because two of them decode a lot of rows."""
    counts, busiest, recent = await asyncio.gather(
        pool.fetchval("SELECT count(*) FROM store"),
        pool.fetchrow("SELECT kind, n FROM summary ORDER BY n DESC, kind LIMIT 1"),
        pool.fetch("SELECT id, kind FROM store ORDER BY id DESC LIMIT 3"))
    return {"rows": counts, "busiest": tuple(busiest), "latest": [r["id"] for r in recent]}


pool = await asyncpg.create_pool(database="guide", min_size=4, max_size=4)

print(await questions(pool))
print("three queries, three connections, one wait")


{'rows': 10000, 'busiest': ('click', 10000), 'latest': [10000, 9999, 9998]}
three queries, three connections, one wait


Three queries, three connections out of a pool of four, one wait. **AsyncConnection** established
that concurrency needs a connection each and **Connection Pools** established where those come from,
and this is the two of them applied.

### The follower: asyncpg, and a watermark rather than a counter

The piece the first look was about. It is woken by a notification and then ignores what the
notification said:


In [5]:
async def follow(seconds, since):
    """asyncpg, because a listener has no autocommit to forget. The watermark does the real work."""
    conn = await asyncpg.connect(database="guide")
    woken = asyncio.Queue()
    await conn.add_listener("store", lambda c, pid, channel, payload: woken.put_nowait(payload))

    seen, wakeups = since, 0
    try:
        async with asyncio.timeout(seconds):
            while True:
                await woken.get()                                   # what it says is not used
                wakeups += 1
                rows = await conn.fetch("SELECT id FROM store WHERE id > $1 ORDER BY id", seen)
                if rows:
                    seen = rows[-1]["id"]
    except TimeoutError:
        pass
    finally:
        await conn.close()
    return wakeups, seen


watermark = counted("SELECT coalesce(max(id), 0) FROM store")
threading.Timer(0.3, load, args=(events(1000, kind="view"),)).start()

wakeups, seen = await follow(seconds=4, since=watermark)
print("woken", wakeups, "time(s), and caught up to id", seen)
print("rows written in that window:", seen - watermark)


woken 1 time(s), and caught up to id 11000
rows written in that window: 1000


One wakeup, a thousand rows, and none of them lost. The notification's payload is never read, which
is the whole design: it is a signal that the table changed, and the table is asked what changed.

### When to reach for which

| The piece | Driver | Why that one |
|---|---|---|
| bulk load | psycopg | `COPY` streaming from Python, with no asyncpg equivalent |
| many small writes | psycopg | pipeline mode, which asyncpg does not have |
| concurrent reads | asyncpg | a connection each, and faster decoding of wide results |
| a listener | asyncpg | no autocommit to forget, and a callback rather than a loop |
| a single query anywhere | either | the measurement in **Which Driver** says it does not matter |

The default for a program with only one of these shapes is one driver, and psycopg is the one that
does everything. Two drivers is a decision to make when the program genuinely has both halves, and
the cost is two connection vocabularies in one codebase, which is not free.

### The whole thing, finished


In [6]:
class EventStore:
    """Four pieces, two drivers, and one watermark that makes the notifications optional."""

    def __init__(self, dsn="dbname=guide", size=4):
        self.dsn, self.size, self.pool = dsn, size, None

    async def startup(self):
        self.pool = await asyncpg.create_pool(self.dsn.replace("dbname=", "postgresql:///"),
                                              min_size=1, max_size=self.size)

    async def shutdown(self):
        await self.pool.close()

    def load(self, rows):                                           # psycopg: COPY
        with psycopg.connect(self.dsn) as conn:
            with conn.cursor().copy("COPY store (kind, payload) FROM STDIN") as copy:
                for row in rows:
                    copy.write_row(row)
            conn.commit()

    def summarize(self):                                            # psycopg: a pipeline
        with psycopg.connect(self.dsn) as conn:
            high = conn.execute("SELECT coalesce(max(id), 0) FROM store").fetchone()[0]
            totals = conn.execute("SELECT kind, count(*) FROM store WHERE id <= %s "
                                  "GROUP BY kind", (high,)).fetchall()
            with conn.pipeline():
                for kind, number in totals:
                    conn.execute("""INSERT INTO summary (kind, n, through) VALUES (%s, %s, %s)
                                    ON CONFLICT (kind) DO UPDATE
                                    SET n = EXCLUDED.n, through = EXCLUDED.through""",
                                 (kind, number, high))
            conn.commit()
        return high

    async def report(self):                                         # asyncpg: concurrent reads
        rows, summarized = await asyncio.gather(
            self.pool.fetchval("SELECT count(*) FROM store"),
            self.pool.fetch("SELECT kind, n, through FROM summary ORDER BY kind"))
        behind = rows - max((row["through"] for row in summarized), default=0)
        return {"rows": rows, "summary": [tuple(r)[:2] for r in summarized], "unsummarized": behind}


store = EventStore()
await store.startup()

store.load(events(500, kind="purchase"))
print("before summarizing:", await store.report())

store.summarize()
print("after summarizing: ", await store.report())

await store.shutdown()


before summarizing: {'rows': 11500, 'summary': [('click', 10000)], 'unsummarized': 1500}
after summarizing:  {'rows': 11500, 'summary': [('click', 10000), ('purchase', 500), ('view', 1000)], 'unsummarized': 0}


`unsummarized` is the number that makes the whole thing operable: it is rows written minus rows the
summary has accounted for, and it is what you put on a dashboard. It goes up when loading outpaces
summarizing and returns to zero when the summary catches up, and no notification was needed to
compute it.

### Where each part came from

| In the event store | What it relies on | The notebook that showed it |
|---|---|---|
| `copy(...)` and `write_row` | `COPY` streaming from Python | **COPY** |
| `with conn.pipeline()` | round trips removed from many small writes | **Pipeline Mode** |
| `ON CONFLICT DO UPDATE` | one statement that inserts or updates | **Connecting and Executing** |
| `asyncio.gather` over a pool | a connection per concurrent query | **AsyncConnection** |
| `asyncpg.create_pool` | connections opened once, not per call | **Connection Pools** |
| `add_listener` and the queue | a callback that cannot block or be awaited | **LISTEN and NOTIFY** |
| `payload jsonb` and `->>` | a column that holds a document | **JSONB** |
| psycopg here and asyncpg there | measurements rather than preferences | **Which Driver** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/17-an-event-store-solutions.ipynb).

**1.** Load a thousand rows with `COPY` and report how many are in the table.


In [7]:
# your code here


**2.** Write the summary for every kind in one pipeline, and print it.


In [8]:
# your code here


**3.** Ask three questions of the store at once through an asyncpg pool.


In [9]:
# your code here


**4.** Count how many notifications a load of five hundred identical rows produces.


In [10]:
# your code here


**5.** Read everything after a watermark and report the new watermark.


In [11]:
# your code here


**6.** Report how many rows the summary has not accounted for yet.


In [12]:
# your code here


## Common errors

### psycopg_pool.PoolTimeout: couldn't get a connection after 1.00 sec


In [13]:
small = psycopg_pool.AsyncConnectionPool("dbname=guide", min_size=2, max_size=2,
                                         timeout=1, open=False)
await small.open(wait=True, timeout=10)


async def slow_report(number):
    async with small.connection() as conn:
        await conn.execute("SELECT pg_sleep(1)")
        return number


await asyncio.gather(*(slow_report(number) for number in range(8)))


PoolTimeout: couldn't get a connection after 1.00 sec

Eight tasks, two connections, and a one second limit on waiting for a turn. Six of the tasks waited
and then gave up.

Nothing is leaked and nothing is broken: the pool is smaller than the concurrency the program asked
for. Either raise `max_size`, having counted it against the server's limit as **Connection Pools**
insisted, or bound the concurrency to match:


In [14]:
turnstile = asyncio.Semaphore(2)                                    # never ask for more than there is


async def politely(number):
    async with turnstile:
        async with small.connection() as conn:
            await conn.execute("SELECT pg_sleep(0.2)")
            return number


start = time.perf_counter()
print("eight tasks, two connections:", await asyncio.gather(*(politely(n) for n in range(8))))
print(f"took {time.perf_counter() - start:.1f}s, and nothing timed out")
await small.close()


eight tasks, two connections: [0, 1, 2, 3, 4, 5, 6, 7]
took 0.8s, and nothing timed out


### psycopg.errors.SerializationFailure: could not serialize access due to concurrent update


In [15]:
with psycopg.connect("dbname=guide", autocommit=True) as setup:
    setup.execute("INSERT INTO summary VALUES ('contended', 0, 0) ON CONFLICT DO NOTHING")


def at_repeatable_read():
    """A connection whose snapshot is taken now, before either writer has changed anything."""
    conn = psycopg.connect("dbname=guide")
    conn.isolation_level = psycopg.IsolationLevel.REPEATABLE_READ
    conn.execute("SELECT n FROM summary WHERE kind = 'contended'").fetchone()
    return conn


first = at_repeatable_read()
first.execute("UPDATE summary SET n = n + 1 WHERE kind = 'contended'")
outcome = []


def second():
    conn = at_repeatable_read()
    try:
        conn.execute("UPDATE summary SET n = n + 10 WHERE kind = 'contended'")   # waits, then fails
        conn.commit()
        outcome.append("committed")
    except psycopg.errors.SerializationFailure as error:
        outcome.append(f"{type(error).__name__}: {error}")
    finally:
        conn.close()


runner = threading.Thread(target=second)
runner.start()
time.sleep(0.5)
first.commit()                                                      # this is what releases the other
runner.join(10)
first.close()
print("the second writer:", outcome)


the second writer: ['SerializationFailure: could not serialize access due to concurrent update']


Two summary writers ran at once and the second one was refused, because at `repeatable read` it had
read a row that the first one changed before it committed. This is not a deadlock and not a bug: it
is the database refusing to produce a result that no serial order could have produced.

The fix is to expect it. A summary writer is idempotent by construction, so retrying is safe, and
the standard shape is a small retry loop:


In [16]:
def summarize_with_retry(attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            return attempt, summarize()[0]
        except psycopg.errors.SerializationFailure:
            time.sleep(0.05 * attempt)
    raise RuntimeError(f"gave up after {attempts} attempts")


attempt, high = summarize_with_retry()
print(f"summarized through id {high} on attempt {attempt}")


summarized through id 11500 on attempt 1


### No error: one notification for a thousand rows


In [17]:
watching = psycopg.connect("dbname=guide", autocommit=True)
watching.execute("LISTEN store")
before = counted("SELECT coalesce(max(id), 0) FROM store")

threading.Timer(0.3, load, args=(events(1000, kind="identical"),)).start()
notifications = list(watching.notifies(timeout=4, stop_after=1000))

after = counted("SELECT coalesce(max(id), 0) FROM store")
print("rows written:  ", after - before)
print("notifications: ", len(notifications))
print("payloads seen: ", {note.payload for note in notifications})


rows written:   1000
notifications:  1
payloads seen:  {'identical'}


The trigger ran a thousand times and `pg_notify` was called a thousand times. PostgreSQL delivers
duplicate notifications raised within one transaction once, and a thousand rows with the same kind
produce a thousand identical notifications.

Distinct payloads are not collapsed, which is worth knowing but is not a fix: it makes the count
depend on how varied your data happens to be, which is worse than knowing it is always wrong:


In [18]:
before = counted("SELECT coalesce(max(id), 0) FROM store")
threading.Timer(0.3, load, args=([("kind %d" % n, '{"n": 1}') for n in range(5)],)).start()

varied = list(watching.notifies(timeout=4, stop_after=5))
print("5 rows, 5 different kinds ->", len(varied), "notifications")
print("so the count depends on the data, which is why nobody should count them")


5 rows, 5 different kinds -> 5 notifications
so the count depends on the data, which is why nobody should count them


### No error: everything sent while the listener was away


In [19]:
watching.execute("UNLISTEN store")                                  # the listener restarts
watermark = counted("SELECT coalesce(max(id), 0) FROM store")

load(events(300, kind="while_away"))                                # and misses all of this

watching.execute("LISTEN store")
print("notifications waiting for it:", len(list(watching.notifies(timeout=1, stop_after=10))))
print("rows it would have missed:   ", counted(
    "SELECT count(*) FROM store WHERE id > %s", (watermark,)))
print("rows the watermark finds:    ", len(counted(
    "SELECT array_agg(id) FROM store WHERE id > %s", (watermark,))))


notifications waiting for it: 0
rows it would have missed:    300
rows the watermark finds:     300


No notifications, three hundred rows, and the watermark finds every one of them. Nothing is stored
by `NOTIFY` and there is no redelivery, so a program that restarts has no way to ask what it missed
except by asking the table.

That is why the follower above never reads the payload, and why the same query serves both cases:
woken or restarted, the question is the same one.


In [20]:
watching.execute("UNLISTEN *")
watching.close()
await pool.close()
print("closed")


closed


## Recap

- An event store is an append-only table, a summary derived from it, and a watermark. Everything
  else in the program is a question of the form "what is new since this id".
- Use psycopg where `COPY` and pipeline mode are: the bulk load and the many small summary writes.
  Neither has an asyncpg equivalent this guide would recommend over them.
- Use asyncpg where concurrency and listening are: a pool, `asyncio.gather`, and a listener with no
  autocommit to forget.
- PostgreSQL collapses identical notifications raised in one transaction into one delivery, so a
  thousand rows can wake a listener once. Never count notifications.
- A notification is a hint that the table changed. Read the table, from the watermark, and the same
  code handles being woken once, woken twice, or not woken at all.
- A pool smaller than the number of concurrent tasks raises `PoolTimeout`. Raise the pool or bound
  the concurrency with a semaphore.
- Two writers at `repeatable read` touching one row give `SerializationFailure`. An idempotent
  writer can simply retry.


## What is next

That is the guide. You started by connecting to a server you had to install, and you have finished
with a program that loads, summarizes, reads concurrently and follows its own table, on two drivers
chosen with measurements.

The natural next step is the SQLAlchemy guide, which puts a mapper over the same server and the same
SQL, and where everything in **Transactions and Errors** and **Connection Pools** turns up again
under different names.


---

&#8592; **Previous:** [Which Driver](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/16-which-driver.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
